In [1]:
import json
import argparse
import librosa
import numpy as np
from pathlib import Path
import warnings


In [2]:
SAMPLE_RATE = 22_050
FRAME_LENGTH = 2_048
HOP_LENGTH = 512
EPSILON = 1e-8

In [3]:
def load_audio(audio_path: str | Path) -> tuple[np.ndarray, int]:
  path=Path(audio_path)
  if not path.is_file():
    raise FileNotFoundError(f"Файл не найден: {path}")
  signal,sample_rate=librosa.load(path,sr=SAMPLE_RATE,mono=True)
  return signal,sample_rate
  #переводит звук в числа понятные для компа


In [4]:
def extract_duration(signal:np.ndarray,sample_rate:int):
  return float(librosa.get_duration(y=signal, sr=sample_rate))
  # просто отдает длительнось в секундах


In [5]:
def extract_rms(signal: np.ndarray) -> np.ndarray:
    """Вычисляет RMS-громкость для каждого короткого фрагмента записи."""
    return librosa.feature.rms(
        y=signal, frame_length=FRAME_LENGTH, hop_length=HOP_LENGTH
    )[0]

In [6]:
def find_speech_frames(rms: np.ndarray) -> np.ndarray:
    """Отмечает кадры, где энергии достаточно, чтобы считать их речью."""
    if not rms.size or float(np.max(rms)) <= EPSILON:
        return np.zeros(rms.size, dtype=bool)
    threshold = max(float(np.percentile(rms, 25)) * 1.5, float(np.max(rms)) * 0.08)
    return rms > threshold

In [7]:
def extract_speech_ratio(speech_frames: np.ndarray) -> float:
    """Доля кадров, которые алгоритм считает активным звуком/речью."""
    return float(np.mean(speech_frames)) if speech_frames.size else 0.0

In [8]:
def safe_mean(values: np.ndarray) -> float:
  values=values[np.isfinite(values)]
  return float(np.mean(values)) if values.size else 0.0

In [9]:
def safe_std(values: np.ndarray) -> float:
  values=values[np.isfinite(values)]
  return float(np.std(values)) if values.size else 0.0

In [10]:
def extract_pause_ratio(speech_frames:np.ndarray) ->float:
  return 1.0 - extract_speech_ratio(speech_frames)

In [11]:
def extract_tempo(signal: np.ndarray, sample_rate: int) -> float:
    """Оценивает число акустических событий в минуту, а не слов в минуту."""
    onset_strength = librosa.onset.onset_strength(
        y=signal, sr=sample_rate, hop_length=HOP_LENGTH
    )
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        tempo = librosa.feature.tempo(
            onset_envelope=onset_strength, sr=sample_rate, hop_length=HOP_LENGTH
        )
    return float(np.ravel(tempo)[0]) if np.size(tempo) else 0.0

In [12]:
def extract_pitch(signal: np.ndarray, sample_rate: int) -> np.ndarray:
    """Ищет основную частоту голоса (pitch) алгоритмом pYIN."""
    pitch, _, _ = librosa.pyin(
        signal,
        fmin=librosa.note_to_hz("C2"),
        fmax=librosa.note_to_hz("C7"),
        sr=sample_rate,
        frame_length=FRAME_LENGTH,
        hop_length=HOP_LENGTH,
    )
    return pitch[np.isfinite(pitch)]

In [13]:
def extract_pitch_mean(pitch: np.ndarray) -> float:
    """Средняя высота голоса в герцах среди звучащих кадров."""
    return safe_mean(pitch)


def extract_pitch_variability(pitch: np.ndarray) -> float:
    """Насколько менялась высота голоса: стандартное отклонение pitch."""
    return safe_std(pitch)


def extract_volume_mean(rms: np.ndarray) -> float:
    """Средняя RMS-громкость записи."""
    return safe_mean(rms)

In [14]:
def extract_volume_variability(rms: np.ndarray) -> float:
    """Относительное изменение громкости во времени."""
    return safe_std(rms) / (safe_mean(rms) + EPSILON)


def extract_jitter_proxy(pitch: np.ndarray) -> float:
    """Учебная оценка нестабильности pitch, это не клинический jitter."""
    if pitch.size < 3:
        return 0.0
    return float(np.mean(np.abs(np.diff(pitch))) / (safe_mean(pitch) + EPSILON))

In [15]:
def extract_shimmer_proxy(rms: np.ndarray, speech_frames: np.ndarray) -> float:
    """Учебная оценка нестабильности амплитуды, это не clinical shimmer."""
    voiced_rms = rms[speech_frames]
    if voiced_rms.size < 3:
        return 0.0
    return float(np.mean(np.abs(np.diff(voiced_rms))) / (safe_mean(voiced_rms) + EPSILON))


def extract_zero_crossing_rate(signal: np.ndarray) -> float:
    """Средняя частота пересечений нуля: грубый признак шумности/резкости."""
    zcr = librosa.feature.zero_crossing_rate(
        signal, frame_length=FRAME_LENGTH, hop_length=HOP_LENGTH
    )[0]
    return safe_mean(zcr)


def extract_spectral_centroid(signal: np.ndarray, sample_rate: int) -> float:
    """Средний спектральный центр, связанный с субъективной яркостью тембра."""
    centroid = librosa.feature.spectral_centroid(
        y=signal, sr=sample_rate, hop_length=HOP_LENGTH
    )[0]
    return safe_mean(centroid)

In [16]:
def analyze_voice(audio_path: str | Path) -> dict[str, float]:
    """Собирает все признаки в одном словаре, удобном для печати или JSON."""
    signal, sample_rate = load_audio(audio_path)
    rms = extract_rms(signal)
    speech_frames = find_speech_frames(rms)
    pitch = extract_pitch(signal, sample_rate)

    return {
        "duration_sec": extract_duration(signal, sample_rate),
        "speech_ratio": extract_speech_ratio(speech_frames),
        "pause_ratio": extract_pause_ratio(speech_frames),
        "estimated_tempo_events_per_min": extract_tempo(signal, sample_rate),
        "pitch_mean_hz": extract_pitch_mean(pitch),
        "pitch_variability_hz": extract_pitch_variability(pitch),
        "volume_rms_mean": extract_volume_mean(rms),
        "volume_variability": extract_volume_variability(rms),
        "jitter_proxy": extract_jitter_proxy(pitch),
        "shimmer_proxy": extract_shimmer_proxy(rms, speech_frames),
        "zero_crossing_rate_mean": extract_zero_crossing_rate(signal),
        "spectral_centroid_mean_hz": extract_spectral_centroid(signal, sample_rate),
    }


def print_features(features: dict[str, float]) -> None:
    """Выводит признаки в читаемой таблице."""
    print("\nАкустические признаки голоса")
    print("-" * 47)
    for name, value in features.items():
        print(f"{name:36} {value:10.4f}")
    print("\nВажно: это измерения звука, а не медицинский диагноз.")


def main() -> None:
    parser = argparse.ArgumentParser(
        description="Учебное извлечение акустических признаков из аудиофайла."
    )
    parser.add_argument("audio", type=Path, help="Путь к WAV, MP3 или другому файлу, который читает librosa.")
    parser.add_argument("--json", type=Path, help="При необходимости сохранить признаки в JSON-файл.")
    args = parser.parse_args()

    features = analyze_voice(args.audio)
    print_features(features)
    if args.json:
        args.json.parent.mkdir(parents=True, exist_ok=True)
        args.json.write_text(json.dumps(features, ensure_ascii=False, indent=2), encoding="utf-8")
        print(f"Результат также сохранён: {args.json}")

In [17]:
!python Voiceanalys.ipynb "/content/audio.mp3"

python3: can't open file '/content/Voiceanalys.ipynb': [Errno 2] No such file or directory


In [18]:
audio_path = "/content/audio.mp3"

features = analyze_voice(audio_path)
print_features(features)

FileNotFoundError: Файл не найден: /content/audio.mp3

In [ ]:
import keras
from keras import layers
FEATURE_NAMES = [
    "duration_sec",
    "speech_ratio",
    "pause_ratio",
    "estimated_tempo_events_per_min",
    "pitch_mean_hz",
    "pitch_variability_hz",
    "volume_rms_mean",
    "volume_variability",
    "jitter_proxy",
    "shimmer_proxy",
    "zero_crossing_rate_mean",
    "spectral_centroid_mean_hz",
]

In [ ]:
def build_model()-> keras.Model:
  normalizer=layers.Normalization(name="normolize_features")
  model=keras.Sequential([
      layers.Input(shape=(len(FEATURE_NAMES),)),
      normalizer,
      layers.Dense(32, activation="relu"),
      layers.Dropout(0.25),
      layers.Dense(16, activation="relu"),
      layers.Dropout(0.15),
      layers.Dense(1, activation="sigmoid")

  ])

  model.compile(
        optimizer="adam",
        loss="binary_crossentropy",
        metrics=["accuracy"],
    )
  return model


In [ ]:
def train_model(
    model: keras.Model,
    features: np.ndarray,
    labels: np.ndarray,
    epochs: int = 40,
) -> keras.callbacks.History:
    """Нормализует признаки и запускает учебное обучение.

    features: таблица формы (число записей, 12).
    labels: нули и единицы, подготовленные исследователем.
    """
    features = np.asarray(features, dtype="float32")
    labels = np.asarray(labels, dtype="float32")

    expected_shape = (features.shape[0], len(FEATURE_NAMES))
    if features.ndim != 2 or features.shape != expected_shape:
        raise ValueError(
            f"Ожидалась таблица формы (число_записей, {len(FEATURE_NAMES)}), "
            f"получено: {features.shape}"
        )
    if labels.shape != (features.shape[0],):
        raise ValueError("Для каждой строки признаков нужна одна метка 0 или 1.")

    normalizer = model.get_layer("normalize_features")
    normalizer.adapt(features)

    return model.fit(
        features,
        labels,
        validation_split=0.2,
        epochs=epochs,
        batch_size=16,
        verbose=2,
    )

model = build_model()
model.summary()
print("\nПризнаки, которые подаются на вход:")
for number, feature_name in enumerate(FEATURE_NAMES, start=1):
  print(f"{number:2}. {feature_name}")
print("\nЭто структура модели. Для обучения нужны реальные размеченные данные.")